In [23]:
import os
import threading
import time
import duckdb
import psutil
import pandas as pd
import polars as pl

import datafusion

In [24]:
pandas_benchamark = False
polars_benchamrk = True
duckdb_benchamrk = False
datafusion_benchmark = False

In [25]:
files = [
    "data/iowa_sales.snappy.parquet",
    "data/iowa_category.snappy.parquet",
    "data/iowa_date.snappy.parquet",
    "data/iowa_store.snappy.parquet",
    "data/iowa_vendor.snappy.parquet",
    "data/iowa_item.snappy.parquet",
]

summary = []

for file in files:
    df = pd.read_parquet(file)

    summary.append(
        {
            "file": file,
            "rows": len(df),
            "columns": len(df.columns),
            "size_mb": round(os.path.getsize(file) / 1024 / 1024, 2),
        }
    )

pd.DataFrame(summary)

,file,rows,columns,size_mb
0,data/iowa_sales.snappy.parquet,1000000,17,41.55
1,data/iowa_category.snappy.parquet,300,3,0.00
2,data/iowa_date.snappy.parquet,5479,18,0.07
3,data/iowa_store.snappy.parquet,2500,8,0.05
4,data/iowa_vendor.snappy.parquet,600,3,0.01
5,data/iowa_item.snappy.parquet,50000,3,0.59


In [26]:
def run_benchmark(func, name):
    process = psutil.Process(os.getpid())

    max_ram = 0
    max_cpu = 0
    running = True

    def monitor():
        nonlocal max_ram, max_cpu, running

        process.cpu_percent()  # warmup

        while running:
            ram = process.memory_info().rss / 1024**3
            cpu = process.cpu_percent()
            max_ram = max(max_ram, ram)
            max_cpu = max(max_cpu, cpu)
            time.sleep(0.1)

    monitor_thread = threading.Thread(target=monitor)
    monitor_thread.start()
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    running = False
    monitor_thread.join()

    return {
        "engine": name,
        "time": round(elapsed, 2),
        "peak_ram_gb": round(max_ram, 2),
        "peak_cpu_pct": round(max_cpu, 0),
        "rows": len(result),
    }

In [27]:
def save_results(results, filename="benchmark_results.csv"):
    df_results = pd.DataFrame(results)

    avg_row = {
        "engine": df_results["engine"].iloc[0] + "_AVG",
        "time": round(df_results["time"].mean(), 2),
        "peak_ram_gb": round(df_results["peak_ram_gb"].mean(), 2),
        "peak_cpu_pct": round(df_results["peak_cpu_pct"].mean(), 0),
        "rows": int(df_results["rows"].mean())
    }

    df_results = pd.concat(
        [df_results, pd.DataFrame([avg_row])],
        ignore_index=True
    )

    df_results.to_csv(
        filename,
        mode="a",
        header=not os.path.exists(filename),
        index=False
    )

    display(df_results)

In [28]:
def pandas_run():

    sales = pd.read_parquet("data/iowa_sales.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    category = pd.read_parquet("data/iowa_category.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    date = pd.read_parquet("data/iowa_date.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    store = pd.read_parquet("data/iowa_store.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    vendor = pd.read_parquet("data/iowa_vendor.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    item = pd.read_parquet("data/iowa_item.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )

    result = (
        sales.merge(category, on="CategoryID")
        .merge(date, on="DateID")
        .merge(store, on="StoreID")
        .merge(vendor, on="VendorID")
        .merge(item, on="ItemID")
        .query("CalendarYear >= 2022")
        .groupby(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            total_sales=("SaleDollars", "sum"),
            bottles_sold=("BottlesSold", "sum"),
            volume_liters=("VolumeSoldLiters", "sum"),
            avg_retail=("StateBottleRetail", "mean"),
            avg_cost=("StateBottleCost", "mean"),
            transactions=("ID", "count"),
        )
        .sort_values(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
                "total_sales",
                "bottles_sold",
            ],
            ascending=[True, True, True, True, True, True, False, False],
        )
    )

    return result

In [29]:
def polars_run():

    sales = pl.scan_parquet("data/iowa_sales.snappy.parquet").drop("Inserted", strict=False)
    category = pl.scan_parquet("data/iowa_category.snappy.parquet").drop(
        "Inserted", strict=False
    )
    date = pl.scan_parquet("data/iowa_date.snappy.parquet").drop("Inserted", strict=False)
    store = pl.scan_parquet("data/iowa_store.snappy.parquet").drop("Inserted", strict=False)
    vendor = pl.scan_parquet("data/iowa_vendor.snappy.parquet").drop(
        "Inserted", strict=False
    )
    item = pl.scan_parquet("data/iowa_item.snappy.parquet").drop("Inserted", strict=False)

    result = (
        sales.join(category, on="CategoryID")
        .join(date, on="DateID")
        .join(store, on="StoreID")
        .join(vendor, on="VendorID")
        .join(item, on="ItemID")
        .filter(pl.col("CalendarYear") >= 2022)
        .group_by(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
            ]
        )
        .agg(
            [
                pl.col("SaleDollars").sum().alias("total_sales"),
                pl.col("BottlesSold").sum().alias("bottles_sold"),
                pl.col("VolumeSoldLiters").sum().alias("volume_liters"),
                pl.col("StateBottleRetail").mean().alias("avg_retail"),
                pl.col("StateBottleCost").mean().alias("avg_cost"),
                pl.col("ID").count().alias("transactions"),
            ]
        )
        .sort(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
                "total_sales",
                "bottles_sold",
            ],
            descending=[False, False, False, False, False, False, True, True],
        )
        .collect()
    )

    result.head(20)
    return result

In [30]:
def duckdb_run():

    result = duckdb.sql("""
    SELECT
        d.CalendarYear,
        s.County,
        s.StoreName,
        v.VendorName,
        c.CategoryName,
        i.ItemName,
        SUM(f.SaleDollars)       AS total_sales,
        SUM(f.BottlesSold)       AS bottles_sold,
        SUM(f.VolumeSoldLiters)  AS volume_liters,
        AVG(f.StateBottleRetail) AS avg_retail,
        AVG(f.StateBottleCost)   AS avg_cost,
        COUNT(f.ID)              AS transactions
    FROM read_parquet('data/iowa_sales.snappy.parquet') f
    JOIN read_parquet('data/iowa_category.snappy.parquet') c
        ON f.CategoryID = c.CategoryID
    JOIN read_parquet('data/iowa_date.snappy.parquet') d
        ON f.DateID = d.DateID
    JOIN read_parquet('data/iowa_store.snappy.parquet') s
        ON f.StoreID = s.StoreID
    JOIN read_parquet('data/iowa_vendor.snappy.parquet') v
        ON f.VendorID = v.VendorID
    JOIN read_parquet('data/iowa_item.snappy.parquet') i
        ON f.ItemID = i.ItemID
    WHERE d.CalendarYear >= 2022
    GROUP BY
        d.CalendarYear,
        s.County,
        s.StoreName,
        v.VendorName,
        c.CategoryName,
        i.ItemName
    ORDER BY
        d.CalendarYear,
        s.County,
        s.StoreName,
        v.VendorName,
        c.CategoryName,
        i.ItemName,
        total_sales DESC,
        bottles_sold DESC
    """).df()

    return result

In [31]:
def datafusion_run():

    ctx = datafusion.SessionContext()

    ctx.register_parquet("sales", "data/iowa_sales.snappy.parquet")
    ctx.register_parquet("category", "data/iowa_category.snappy.parquet")
    ctx.register_parquet("date", "data/iowa_date.snappy.parquet")
    ctx.register_parquet("store", "data/iowa_store.snappy.parquet")
    ctx.register_parquet("vendor", "data/iowa_vendor.snappy.parquet")
    ctx.register_parquet("item", "data/iowa_item.snappy.parquet")

    result = ctx.sql("""
    SELECT
        d."CalendarYear",
        s."County",
        s."StoreName",
        v."VendorName",
        c."CategoryName",
        i."ItemName",
        SUM(f."SaleDollars")       AS total_sales,
        SUM(f."BottlesSold")       AS bottles_sold,
        SUM(f."VolumeSoldLiters")  AS volume_liters,
        AVG(f."StateBottleRetail") AS avg_retail,
        AVG(f."StateBottleCost")   AS avg_cost,
        COUNT(f."ID")              AS transactions
    FROM sales f
    JOIN category c ON f."CategoryID" = c."CategoryID"
    JOIN date d     ON f."DateID" = d."DateID"
    JOIN store s    ON f."StoreID" = s."StoreID"
    JOIN vendor v   ON f."VendorID" = v."VendorID"
    JOIN item i     ON f."ItemID" = i."ItemID"
    WHERE d."CalendarYear" >= 2022
    GROUP BY
        d."CalendarYear",
        s."County",
        s."StoreName",
        v."VendorName",
        c."CategoryName",
        i."ItemName"
    ORDER BY
        d."CalendarYear",
        s."County",
        s."StoreName",
        v."VendorName",
        c."CategoryName",
        i."ItemName",
        total_sales DESC,
        bottles_sold DESC
    """).to_pandas()

    return result

In [32]:
results = []
if pandas_benchamark:
    for _ in range(5):
        results.append(run_benchmark(pandas_run, "Pandas"))
    save_results(results)

results = []
if polars_benchamrk:
    for _ in range(5):
        results.append(run_benchmark(polars_run, "Polars"))
    save_results(results)

results = []
if duckdb_benchamrk:
    for _ in range(5):
        results.append(run_benchmark(duckdb_run, "DuckDB"))
    save_results(results)


results = []
if datafusion_benchmark:
    for _ in range(5):
        results.append(run_benchmark(datafusion_run, "DataFusion"))
    save_results(results)

,engine,time,peak_ram_gb,peak_cpu_pct,rows
0,Polars,0.13,1.08,606.0,333789
1,Polars,0.09,1.09,0.0,333789
2,Polars,0.08,1.29,0.0,333789
3,Polars,0.08,1.40,0.0,333789
4,Polars,0.08,1.49,0.0,333789
5,Polars_AVG,0.09,1.27,121.0,333789


In [33]:
pd.read_csv("benchmark_results.csv")

,engine,time,peak_ram_gb,peak_cpu_pct,rows
0,DuckDB,0.33,0.90,716.0,333789
1,DuckDB,0.32,0.96,778.0,333789
2,DuckDB,0.32,0.97,747.0,333789
3,DuckDB,0.33,0.84,904.0,333789
4,DuckDB,0.35,0.72,702.0,333789
5,DuckDB_AVG,0.33,0.88,769.0,333789
6,Polars,0.13,1.08,606.0,333789
7,Polars,0.09,1.09,0.0,333789
8,Polars,0.08,1.29,0.0,333789
9,Polars,0.08,1.40,0.0,333789
